# Identity Logs Endpoint Example

This notebook demonstrates how to call the endpoint for retrieving Okta logs via the Identity module.

## Setup

We'll initialize our `SyncApiClientFactory` and create an instance of the `IdentityLogsApi`. Make sure your environment variables or equivalent configuration is present and includes an Identity (LUSID) token or credentials.

In [ ]:
# Import Libraries
from datetime import datetime, timedelta
from pprint import pprint
import os

import finbourne.sdk.services.identity as li
from finbourne.sdk.extensions import SyncApiClientFactory
from finbourne.sdk.exceptions import ApiException

# Use the finbourne SyncApiClientFactory to build Api instances with a configured api client
# By default this will read config from environment variables
# Then from a secrets.json file found in the current working directory

secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

api_client_factory = SyncApiClientFactory(
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)

# Create an instance of the API class
api_instance = api_client_factory.build(li.IdentityLogsApi)

## 1. Retrieve Logs without Parameters

We can call `get_logs` with no parameters, which will return all logs (up to the server-defined limits or defaults). In practice, you would often filter by date or paginate for performance.


In [ ]:
try:
    api_response = api_instance.list_logs()
    pprint(api_response)

except ApiException as e:
    print("Exception when calling IdentityLogsApi->list_logs: %s\n" % e)


## 2. Retrieve Logs with Date Parameters

We can pass in parameters such as `okta_since` and `okta_until` to narrow down the time window. These parameters expect ISO8601 strings.

Example curl equivalent:

`curl -X GET "https://<your-domain>.lusid.com/identity/api/logs?oktaSince=2025-03-01T00%3A00%3A00Z&oktaUntil=2025-03-02T00%3A00%3A00Z"
-H "Authorization: Bearer <your-API-access-token>"`

Below, we replicate this in Python using the Identity SDK.

In [ ]:
# Get the logs in the last 2 hours
okta_until = datetime.now().isoformat()
okta_since = (datetime.now() - timedelta(hours=2)).isoformat()

try:
    api_response = api_instance.list_logs(okta_since=okta_since, okta_until=okta_until)
    pprint(api_response)

except ApiException as e:
    print("Exception when calling IdentityLogsApi->list_logs: %s\n" % e)


## 3. Example of retrieving logs using all parameters

Below is an example of using the Identity SDK to retrieve logs using all available parameters

In [ ]:
okta_until = datetime.now().isoformat() # Lower bound of log events published property
okta_since = (datetime.now() - timedelta(hours=2)).isoformat() # Upper bound of log events published property
okta_filter = 'actor.id pr' # Okta filter expression 
okta_query = 'AppUser' # Filters log events results by one or more case insensitive keywords
okta_limit = 200 # Max number of results returned
okta_sort_order = 'ASCENDING' # Order of events by published property. Either ASCENDING or DESCENDING 

try:
    api_response = api_instance.list_logs(okta_since=okta_since, okta_until=okta_until)
    pprint(api_response)

except ApiException as e:
    print("Exception when calling IdentityLogsApi->list_logs: %s\n" % e)